# Chapter 19 — Tiling and Vectorization

Part 7: Kernel construction and optimization.

**Ladder:** 1-D tile simulation (Method 01) → tiled 1-D/2-D copies and a 4-elements-per-thread vector kernel (Method 02) → register-blocked rows, shared-memory-style reuse, and a two-stage software pipeline (Method 03).

**Thread.**

- **Recurring idea:** a tile is a rectangle of work with three properties — a mask, a thread complement, and a memory-transaction width. Every section changes one of them.
- **Chain:** 1-D tile → 2-D tile → 3-D tile → block size → tile size → warps per program → elements per thread → vectorization → register blocking → shared-memory reuse → double buffering → software pipelining.

**Study tip.** Fix the running example: a $5\times7$ matrix cut into $2\times4$ tiles. Compute the grid, the tile volume, and the elements per thread by hand before reading any kernel.

| quantity | calculation | value |
|---|---|---|
| grid rows | $\lceil 5/2\rceil$ | 3 |
| grid columns | $\lceil 7/4\rceil$ | 2 |
| programs | $3\cdot2$ | 6 |
| elements per tile | $2\cdot4$ | 8 |
| padded elements | $6\cdot8$ | 48 |
| live elements | $5\cdot7$ | 35 |
| tail waste | $1-35/48$ | 27.1% |

- **Warp check:** a warp is 32 lanes, so an 8-element tile gives $8/32=0.25$ element per thread with one warp and $8/64=0.125$ with two. Both are below one; most lanes have no work.
- **Design rule:** a tile should be at least as large as the warps that consume it.

**Learning objectives**

- Compute the grid $\lceil M/B_M\rceil\lceil N/B_N\rceil$, the tile volume $V=B_MB_N$, and the elements per thread $e=V/(32W)$.
- Draw a 1-D, 2-D, or 3-D tile map and mark the masked tail lanes.
- Distinguish a **block size** (one extent) from a **tile size** (the volume or its byte count).
- Choose a vector width from the dtype and explain the 16-byte lane transaction.
- Explain register blocking, shared-memory reuse, double buffering, and software pipelining, and recognize their Triton knobs.
- Diagnose the common tiling failures: tail waste, idle lanes, register spill, and unaligned loads.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch18_kernel_construction/18_complete.ipynb) · [Next →](../ch20_fusion/20_complete.ipynb)

**Estimated time:** 75–95 min · **Difficulty:** Advanced

**Prerequisites:** Chapter 12 (layouts), Chapter 13 (memory optimization), Chapter 18 (kernel construction), and comfort with `tl.arange`, masks, and `tl.load`/`tl.store`.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)


## Concept diagrams

### 1-D tiling

```text
N = 13, BLOCK = 4, P = ceil(13/4) = 4

 flat index:  0  1  2  3 | 4  5  6  7 | 8  9 10 11 | 12 13 14 15
 tile:        [  pid 0  ] [  pid 1  ] [  pid 2  ] [  pid 3  ]
 mask:         T  T  T  T   T  T  T  T   T  T  T  T    T  F  F  F
 live:           0..3         4..7         8..11         12
```

One program owns one contiguous chunk. The last tile is padded and masked.

### 2-D tiling

```text
M = 5, N = 7, B_M = 2, B_N = 4  ->  grid = (3, 2), 6 programs

             n = 0..3      n = 4..6
 m = 0,1     (0,0)         (0,1)
 m = 2,3     (1,0)         (1,1)
 m = 4       (2,0)         (2,1)
```

The row tile and the column tile are independent grid axes. The tail is masked on
both axes.

### 3-D tiling

```text
x[B, M, N] with tile (B_M, B_N)

 grid axis 0: b   in 0 .. B-1
 grid axis 1: i   in 0 .. ceil(M/B_M)-1
 grid axis 2: j   in 0 .. ceil(N/B_N)-1

 address(b, m, n) = b*M*N + m*N + n
```

The batch (or head) axis stays outside the spatial tile so the inner pair remains
contiguous.

### Block size versus tile size

```text
block size B_M = 2          one extent
block size B_N = 4          one extent
tile size  V   = 2 * 4 = 8  elements
tile bytes     = 8 * 4 = 32 bytes (fp32)

B_M and B_N are knobs; V is the consequence.
```


### Warps, threads, and elements

```text
tile V = 256, num_warps W = 2

 threads      = 32 * W          = 64
 elements/thr = V / (32 * W)    = 4
 lane t owns  = 4t, 4t+1, 4t+2, 4t+3   (a contiguous group when aligned)
 warp 0       = threads  0..31
 warp 1       = threads 32..63
```

### Vectorized transaction

```text
one lane:   4 fp32 = 16 bytes = one 128-bit vector load
one warp:   32 lanes * 16 B = 512 B = four 128-B sectors
```

The vector width is a dtype property: $16/b$ elements per lane.

### Register blocking

```text
one thread owns an r_m x r_n register tile

        n ->
      +----+----+----+----+
 m    | a  | a  | a  | a  |   one A value feeds r_n outputs
 |    +----+----+----+----+
 v    | a  | a  | a  | a  |
      +----+----+----+----+

 accumulators/thread = r_m * r_n
```

### Double buffer

```text
serial:      L0 C0 L1 C1 L2 C2 L3 C3                8 units
pipelined:   L0 | C0/L1 | C1/L2 | C2/L3 | C3        5 units

one buffer:  [ load k ][ compute k ]   (wait for the load)
two buffers: [ load k+1 ] runs while [ compute k ] runs
```


In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, cdiv, check, require_gpu, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU)


## Part I — Mathematics and concepts

- **Section shape:** Connection → Rule → Formula → Example → Where → Triton → Failure → Bridge.
- **Running example:** the $5\times7$ matrix with $2\times4$ tiles unless a section needs a different shape.
- **Reading order:** the sections follow the ladder from one flat range to pipelined reuse.

### 19.1 1-D tiling

- **Connection.** Opens the ladder: the simplest partition of a flat index range; every later tile reuses this mask pattern.
- **Rule.** A tile of $B$ elements partitions $[0,N)$ into $P=\lceil N/B\rceil$ programs, and program $p$ owns $I_p=\{pB,\dots,pB+B-1\}\cap[0,N)$.
- **Formula.** $P=\left\lceil\frac{N}{B}\right\rceil$.
- **Example.** For $N=13$ and $B=4$, $P=\lceil13/4\rceil=4$; the last program owns only index $12$.

  | pid | offsets | mask | live |
  |---|---|---|---|
  | 0 | 0,1,2,3 | T T T T | 0–3 |
  | 1 | 4,5,6,7 | T T T T | 4–7 |
  | 2 | 8,9,10,11 | T T T T | 8–11 |
  | 3 | 12,13,14,15 | T F F F | 12 |

- **Where.** Contiguous elementwise work — copy, scale, bias add, activation — and the inner loop of a row-major 2-D kernel.
- **Triton.** `offs = pid * BLOCK + tl.arange(0, BLOCK)`, then `mask = offs < n`; keep `BLOCK` a power of two so `tl.arange` is legal.
- **Failure.** Omitting the mask reads and writes past the buffer; a `BLOCK` smaller than $32W$ leaves lanes idle (see §19.7).
- **Bridge.** The next section adds a second axis, so the mask becomes an outer product.


### 19.2 2-D tiling

- **Connection.** Builds on 19.1 by applying the same 1-D mask independently to rows and columns.
- **Rule.** Two grid axes own a row extent $B_M$ and a column extent $B_N$; the tile is their Cartesian product.
- **Formula.** $P=\left\lceil\frac{M}{B_M}\right\rceil\left\lceil\frac{N}{B_N}\right\rceil$, and tile $(i,j)=\{iB_M,\dots,(i+1)B_M-1\}\times\{jB_N,\dots,(j+1)B_N-1\}$.
- **Example.** For the $5\times7$ matrix with $B_M=2$ and $B_N=4$, $P=3\cdot2=6$; the six tiles cover $8+6+8+6+4+3=35$ live elements against a padded footprint of $6\cdot8=48$.
- **Where.** Images, feature maps, attention score blocks, and GEMM output tiles all have row/column structure.
- **Triton.** `rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)`, `cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)`, and mask `rows[:, None] < M` together with `cols[None, :] < N`; launch `(cdiv(M, BM), cdiv(N, BN))`.
- **Failure.** Masking only one axis touches out-of-range memory on the other; a tile wider than the innermost stride breaks coalescing.
- **Bridge.** A batch axis turns the two grid axes into three without changing the inner tile.


### 19.3 3-D tiling

- **Connection.** Extends 19.2 with a leading batch axis; the row/column tile is unchanged.
- **Rule.** The batch/head axis is its own program axis, so the inner pair stays contiguous.
- **Formula.** $P=B\left\lceil\frac{M}{B_M}\right\rceil\left\lceil\frac{N}{B_N}\right\rceil$, with address $\text{addr}(b,m,n)=bMN+mN+n$.
- **Example.** For $B=3$, $M=9$, $N=17$, $B_M=2$, $B_N=4$: $P=3\cdot5\cdot5=75$; the next code cell verifies that the tiles cover all $459$ cells once.
- **Where.** Batched images, per-head attention, grouped GEMM, and per-sample normalization.
- **Triton.** `tl.program_id(0)` is the batch, `(1)` rows, `(2)` columns; form `base = b * M * N` and add the 2-D offsets; promote to `int64` when $BMN$ can exceed $2^{31}$.
- **Failure.** Folding the batch axis into the row index loses the contiguous inner pair and can overflow a 32-bit offset.
- **Bridge.** The remaining sections keep the grid fixed and tune its three knobs: block size, tile size, and warp count.


In [ ]:
def tile_map_3d(B, M, N, BM, BN):
    """Return (b, pid_m, pid_n, live) for every tile in a batch of matrices."""
    tiles = []
    for b in range(B):
        for pm in range(cdiv(M, BM)):
            for pn in range(cdiv(N, BN)):
                r0, c0 = pm * BM, pn * BN
                r1, c1 = min(r0 + BM, M), min(c0 + BN, N)
                tiles.append((b, pm, pn, (r1 - r0) * (c1 - c0)))
    return tiles


def test_3d_tile_map():
    """CPU — the 3-D grid covers the batch exactly once."""
    B, M, N, BM, BN = 3, 9, 17, 2, 4
    assert (B, cdiv(M, BM), cdiv(N, BN)) == (3, 5, 5)

    tiles = tile_map_3d(B, M, N, BM, BN)
    assert len(tiles) == 75
    assert sum(t[3] for t in tiles) == B * M * N == 459

    cover = torch.zeros((B, M, N), dtype=torch.int32)
    for b, pm, pn, _ in tiles:
        r0, c0 = pm * BM, pn * BN
        r1, c1 = min(r0 + BM, M), min(c0 + BN, N)
        cover[b, r0:r1, c0:c1] += 1
    assert int(cover.min()) == 1
    assert int(cover.max()) == 1

    b, m, n = 2, 8, 16
    assert b * M * N + m * N + n == 458
    assert B * M * N - 1 == 458


test_3d_tile_map()


### 19.4 Block size

- **Connection.** Follows 19.1–19.3: the grid formula is fixed, and $B$ is the first extent to sweep.
- **Rule.** A block size is one tile extent, not the whole tile.
- **Formula.** $B\in\{B_M,B_N,B_K\}$ and $P=\left\lceil\frac{N}{B}\right\rceil$.
- **Example.** With $N=1000$:

  | $B$ | $P=\lceil1000/B\rceil$ | last tile |
  |---:|---:|---|
  | 128 | 8 | 104 live |
  | 256 | 4 | 232 live |
  | 512 | 2 | 488 live |

- **Where.** Sets the launch grid and the per-program working set; sweep it first.
- **Triton.** Declare `BLOCK: tl.constexpr` and compute the grid with `triton.cdiv(n, BLOCK)`; keep it a power of two.
- **Failure.** A non-power-of-two `BLOCK` is rejected by `tl.arange`; too large raises register pressure, too small multiplies launch and loop overhead.
- **Bridge.** The product of the block sizes is the tile size, which is the resource that actually has to fit.


### 19.5 Tile size

- **Connection.** Builds on 19.4 by multiplying the block sizes into the unit that consumes registers or shared memory.
- **Rule.** The tile is the product of its extents; its byte count is that product times the element size.
- **Formula.** $V=\prod_i B_i$ and $\text{bytes}=V\,b$.
- **Example.**

  | tile | dtype | $V$ | bytes |
  |---|---:|---:|---:|
  | $2\times4$ | fp32 | 8 | 32 B |
  | $32\times32$ | fp32 | 1024 | 4 KiB |
  | $64\times64$ | fp16 | 4096 | 8 KiB |
  | $128\times128$ | fp16 | 16384 | 32 KiB |

- **Where.** Determines whether the program fits in registers or shared memory.
- **Triton.** Tile shape drives register allocation; increase `num_warps` as $V$ grows so that $e=V/(32W)$ stays in $[1,8]$.
- **Failure.** Too large spills to local memory; too small cannot fill a warp.
- **Bridge.** The tile volume and the warp count together determine the per-thread work.


### 19.6 Warps per program

- **Connection.** Pairs with 19.5: the tile volume is fixed, and the warp count divides it among threads.
- **Rule.** A program with `num_warps=W` has $32W$ threads; more warps cooperate on a larger tile but occupy more of an SM.
- **Formula.** $\text{threads/program}=32W$ and $\text{resident programs}\approx\frac{\text{warps/SM}}{W}$.
- **Example.**

  | $W$ | threads/program | $e$ for $V=256$ |
  |---:|---:|---:|
  | 1 | 32 | 8 |
  | 2 | 64 | 4 |
  | 4 | 128 | 2 |
  | 8 | 256 | 1 |

- **Where.** The scheduling unit; it changes the layout and per-thread work, never the logical result.
- **Triton.** Pass `num_warps=` at launch; compare results across values before trusting a choice.
- **Failure.** Too many warps for a small tile gives $e\lt 1$ and idle lanes; too few warps for a large tile gives many elements per thread and spills.
- **Bridge.** The quotient $V/(32W)$ is the single number that predicts both problems.


### 19.7 Elements per thread

- **Connection.** Divides the 19.5 tile volume by the 19.6 thread count.
- **Rule.** Each thread owns $e$ elements; $e$ must be at least one, and a fractional value means idle lanes.
- **Formula.** $e=\frac{V}{32W}=\frac{\prod_i B_i}{32W}$.
- **Example.**

  | $V$ | $W$ | threads | $e$ | interpretation |
  |---:|---:|---:|---:|---|
  | 8 | 1 | 32 | 0.25 | 3 of 4 lanes idle |
  | 8 | 2 | 64 | 0.125 | 7 of 8 lanes idle |
  | 256 | 2 | 64 | 4 | one 16-B vector per lane |
  | 4096 | 8 | 256 | 16 | heavy register use |

- **Where.** The register-pressure and vectorization knob, and the test of whether the tile fills its warps.
- **Triton.** Choose $W$ so that $e$ is an integer in $[1,8]$; make the tile volume a multiple of $32W$.
- **Failure.** Fractional $e$ leaves lanes idle; very large $e$ spills.
- **Bridge.** Once $e$ is an integer, the next question is how many of those elements can be packed into one vector transaction.


### 19.8 Vectorization

- **Connection.** Turns the per-thread element count from 19.7 into memory transactions.
- **Rule.** A lane can issue at most 128 bits, so the vector width is a dtype property.
- **Formula.** $v=\frac{128}{8b}=\frac{16}{b}$, with $vb\le16$ bytes per lane and $32vb\le512$ bytes per warp.
- **Example.**

  | dtype | $b$ | $v=16/b$ | lane bytes | warp bytes |
  |---|---:|---:|---:|---:|
  | int8 | 1 | 16 | 16 | 512 |
  | fp16/bf16 | 2 | 8 | 16 | 512 |
  | fp32 | 4 | 4 | 16 | 512 |
  | fp64 | 8 | 2 | 16 | 512 |

  For the 1-D tile $B=256$ of fp32 with $e=4$: $v=4$, each lane loads 16 B, and the grid is $\lceil N/256\rceil$. With a contiguous tile the compiler packs the $e$ elements of a lane into $e/v$ transactions; the fully vectorized case is $e=v$, so the vector width equals the elements per thread.
- **Where.** The main lever on memory-bound kernels: coalesced, aligned, contiguous access.
- **Triton.** Use contiguous offsets and aligned bases; `tl.max_contiguous(tl.multiple_of(offs, v), v)` states the guarantee; keep $e\ge v$.
- **Failure.** An unaligned base or a stride between lanes splits one vector into several transactions; a non-contiguous access falls back to scalar loads.
- **Bridge.** Vectorization describes one thread; the next section describes how one thread reuses several loaded values.


### 19.9 Register blocking

- **Connection.** Builds on 19.7 and 19.8: the thread already owns $e$ elements; register blocking shapes them into an $r_m\times r_n$ output tile.
- **Rule.** Each thread accumulates an $r_m\times r_n$ block, reusing one loaded $A$ value $r_n$ times and one loaded $B$ value $r_m$ times.
- **Formula.** $\text{accumulators/thread}=r_mr_n$ and $\text{tile volume}=32W\,r_mr_n$.
- **Example.** With $r_m=r_n=4$ and $W=2$: $V=64\cdot16=1024$ and $\text{accumulators/thread}=16$.
- **Where.** Per-thread reuse without returning to memory; the per-thread analogue of a larger tile.
- **Triton.** A 2-D tile already gives each thread several elements; explicit row loops or `tl.dot` increase $r_mr_n$; keep the accumulator count under the register budget.
- **Failure.** Too many accumulators spills to local memory; too few gives low reuse and a memory-bound kernel even when the arithmetic could be reused.
- **Bridge.** Reuse at thread scope is registers; reuse at block scope is shared memory.


### 19.10 Shared-memory reuse

- **Connection.** Lifts the 19.9 register-reuse idea from one thread to a whole block.
- **Rule.** Shared memory is a per-block scratchpad; many threads reuse one loaded panel, which is why a tiled GEMM can become compute-bound.
- **Formula.** $Q_{\text{smem}}=s\,(B_MB_K+B_KB_N)\,b$ for $s$ pipeline stages.
- **Example.** With $B_M=B_N=B_K=32$, fp16 ($b=2$), and $s=2$: $Q_{\text{smem}}=2(1024+1024)\cdot2=8192$ B $=8$ KiB.
- **Where.** Panel reuse in GEMM and any blocked kernel whose operands are shared by many outputs.
- **Triton.** Triton does not expose shared memory directly; `tl.dot` stages its operands through it, and `num_stages` multiplies the staging buffers; for elementwise kernels, demonstrate reuse with a broadcast row loaded once per tile row.
- **Failure.** Too many stages overflow the SM's shared-memory budget; a layout that maps lanes to the same bank causes bank conflicts.
- **Bridge.** A staging buffer is only useful if the next load can run while the current panel computes.


### 19.11 Double buffering

- **Connection.** Puts the 19.10 staging buffer to work by overlapping the next load with the current compute.
- **Rule.** For $T$ panels, a serial schedule waits for every load; a two-buffer schedule prefetches the next panel.
- **Formula.** $T_{\text{serial}}=2T$, $T_{\text{2-stage}}=T+1$, and $S=\frac{2T}{T+1}$.
- **Example.** For $T=4$: serial $=8$, pipelined $=5$, $S=8/5=1.6$.

  ```text
  serial:      L0 C0 L1 C1 L2 C2 L3 C3                8 units
  pipelined:   L0 | C0/L1 | C1/L2 | C2/L3 | C3        5 units
  ```

  One buffer waits for the load; two buffers let `load k+1` run while `compute k` runs.
- **Where.** Memory latency is hundreds of cycles and arithmetic is a few, so overlap keeps the arithmetic unit busy.
- **Triton.** `num_stages=2` asks for two buffers; verify that the result is unchanged for `num_stages=1,2,3`.
- **Failure.** Too many stages consume shared memory; a loop-carried dependency prevents the compiler from moving the load ahead.
- **Bridge.** The compiler can build this schedule automatically from an independent loop.


### 19.12 Software pipelining

- **Connection.** Generalizes 19.11 from a hand-written look-ahead to the compiler's loop transformation.
- **Rule.** A pipelined loop has a prologue, a steady state, and an epilogue; the steady state does one tile per iteration.
- **Formula.** With $s$ stages: $s-1$ prologue loads $\to$ steady state (one tile/iteration) $\to$ $s-1$ epilogue computes.
- **Example.** With $s=2$ and $T=4$ panels, the schedule is `L0 | C0/L1 | C1/L2 | C2/L3 | C3`: two stages, five steps.
- **Where.** Independent loop iterations over tiles: GEMM K-panels, long elementwise reductions, and persistent kernels.
- **Triton.** Write the loop over independent tiles and pass `num_stages`; do not serialize iterations through a scalar that depends on the previous store.
- **Failure.** A stage mismatch (more buffers than loop iterations) wastes memory; a dependency through memory forces a wait and removes the overlap.
- **Bridge.** The Part I ladder is complete: tile → block → warp → element → vector → register reuse → shared reuse → overlap.

**Where this leads.** Chapter 20 fuses these tiles into single kernels; the same grid, mask, and vector rules decide whether fusion saves traffic or just adds register pressure.


## Chapter map

| § | Topic | Formula | Implication | Kernel/demo |
|---|---|---|---|---|
| **19.1** | 1-D tiling | $P=\lceil N/B\rceil$ | one program per contiguous chunk; mask the tail | `scale1d_kernel` |
| **19.2** | 2-D tiling | $P=\lceil M/B_M\rceil\lceil N/B_N\rceil$ | two grid axes; mask rows and columns | `copy2d_kernel` |
| **19.3** | 3-D tiling | $P=B\lceil M/B_M\rceil\lceil N/B_N\rceil$ | batch stays a separate axis | tile-map arithmetic |
| **19.4** | Block size | $P=\lceil N/B\rceil$ | sweep the first knob | block-size table |
| **19.5** | Tile size | $V=\prod_i B_i$ | the tile must fit registers/SMEM | worked arithmetic |
| **19.6** | Warps per program | $32W$ | changes per-thread work, not the answer | `row_affine_kernel` |
| **19.7** | Elements per thread | $e=V/(32W)$ | keep $e$ an integer in $[1,8]$ | worked table |
| **19.8** | Vectorization | $v=16/b$ | contiguous and aligned | `vec4_scale_kernel` |
| **19.9** | Register blocking | $r_mr_n$ accumulators | per-thread reuse | `row_affine_kernel` |
| **19.10** | Shared-memory reuse | $s(B_MB_K+B_KB_N)b$ | Triton stages via `tl.dot` | `row_broadcast_kernel` |
| **19.11** | Double buffering | $T+1$ vs $2T$ | overlap load and compute | `pipeline_kernel` |
| **19.12** | Software pipelining | prologue/steady/epilogue | `num_stages` | `pipeline_kernel` |
| **★** | Worked example | grid $(3,2)$, $V=8$, $e=0.25$ | all counts checked by asserts | assert cell |


## Worked numerical example

- **A. The $5\times7$ matrix with $2\times4$ tiles.**
  - $M=5$, $N=7$, $B_M=2$, $B_N=4$.
  - $P=\lceil5/2\rceil\lceil7/4\rceil=3\cdot2=6$.
  - Tile $(p_m,p_n)$ owns rows $p_mB_M,\dots,p_mB_M+B_M-1$ and columns $p_nB_N,\dots,p_nB_N+B_N-1$, clipped to the matrix.

  | program $(p_m,p_n)$ | rows | columns | live cells |
  |---|---|---|---|
  | (0,0) | 0,1 | 0,1,2,3 | 8 |
  | (0,1) | 0,1 | 4,5,6 | 6 |
  | (1,0) | 2,3 | 0,1,2,3 | 8 |
  | (1,1) | 2,3 | 4,5,6 | 6 |
  | (2,0) | 4 | 0,1,2,3 | 4 |
  | (2,1) | 4 | 4,5,6 | 3 |

  - Padded footprint $6\cdot8=48$; live footprint $5\cdot7=35$.
  - Tail waste $=1-35/48=13/48\approx27.1\%$.

- **B. Elements per thread.**
  - Tile volume $V=8$, so $e=8/(32W)$.

  | $W$ | threads $32W$ | $e$ for $V=8$ | verdict |
  |---:|---:|---:|---|
  | 1 | 32 | 0.25 | 3 of 4 lanes idle |
  | 2 | 64 | 0.125 | 7 of 8 lanes idle |
  | 4 | 128 | 0.0625 | 15 of 16 lanes idle |

  - Two warps are **64** threads on NVIDIA (one warp is 32 lanes), not 128.
  - The "128 threads, one element per thread" case needs a 128-element tile and 4 warps: $8\times16$ gives $128/(4\cdot32)=1$.
  - Equivalently, $8\times8=64$ with 2 warps gives $64/64=1$.
  - Either way, the $2\times4$ tile is too small: it cannot fill even one warp.

- **C. The 1-D fp32 tile.**
  - For $B=256$, $b=4$, $e=4$: threads $=B/e=64=2$ warps, $v=128/(8\cdot4)=4$ fp32 per lane.
  - Each lane moves $4\cdot4=16$ bytes, one 128-bit vector; a warp moves $32\cdot16=512$ bytes.
  - The grid is $\lceil N/256\rceil$, which is $4$ for $N=1024$.

- **D. Double-buffer timeline.**
  - With $T=4$ panels and two stages, serial is $2T=8$ units and pipelined is $T+1=5$.

  ```text
  serial:      L0 C0 L1 C1 L2 C2 L3 C3                8
  pipelined:   L0 | C0/L1 | C1/L2 | C2/L3 | C3        5
               ^    ^       ^       ^
               |    |       |       +-- L3 overlaps C2
               |    |       +---------- L2 overlaps C1
               |    +------------------ L1 overlaps C0
               +----------------------- prologue
  ```

  - The speedup model is $S=2T/(T+1)=8/5=1.6$; it is a schedule model, not a measured benchmark.


In [ ]:
from ttools import cdiv


def tile_map(M, N, BM, BN):
    """Return (pid_m, pid_n, r0, r1, c0, c1, live) for every 2-D tile."""
    tiles = []
    for pm in range(cdiv(M, BM)):
        for pn in range(cdiv(N, BN)):
            r0, c0 = pm * BM, pn * BN
            r1, c1 = min(r0 + BM, M), min(c0 + BN, N)
            tiles.append((pm, pn, r0, r1, c0, c1, (r1 - r0) * (c1 - c0)))
    return tiles


def test_worked_example():
    """Every count in the worked example is checked silently."""
    M, N, BM, BN = 5, 7, 2, 4
    grid = (cdiv(M, BM), cdiv(N, BN))
    assert grid == (3, 2)

    V = BM * BN
    assert V == 8
    tiles = tile_map(M, N, BM, BN)
    assert len(tiles) == 6
    live = [t[-1] for t in tiles]
    assert live == [8, 6, 8, 6, 4, 3]
    assert sum(live) == 35
    assert len(tiles) * V == 48
    assert abs((1 - 35 / 48) - 13 / 48) < 1e-12

    cover = torch.zeros((M, N), dtype=torch.int32)
    for _, _, r0, r1, c0, c1, _ in tiles:
        cover[r0:r1, c0:c1] += 1
    assert int(cover.min()) == 1
    assert int(cover.max()) == 1
    assert int(cover.sum()) == 35

    # Section tables.
    assert cdiv(13, 4) == 4
    assert 3 * cdiv(9, 2) * cdiv(17, 4) == 75
    assert [cdiv(1000, b) for b in (128, 256, 512)] == [8, 4, 2]
    assert [1000 - (cdiv(1000, b) - 1) * b for b in (128, 256, 512)] == [104, 232, 488]
    assert [2 * 4, 32 * 32, 64 * 64, 128 * 128] == [8, 1024, 4096, 16384]
    assert 2 * 4 * 4 == 32
    assert 64 * 64 * 2 == 8192
    assert 128 * 128 * 2 == 32768
    assert 32 * 2 == 64 and 64 * 16 == 1024
    assert (64 * 64 + 64 * 64) * 2 == 16384
    assert 2 * 16384 == 32768
    assert 3 * 16384 == 49152

    # Warp arithmetic, 32 lanes per warp.
    assert 1 * 32 == 32 and 2 * 32 == 64 and 4 * 32 == 128
    assert V / (1 * 32) == 0.25
    assert V / (2 * 32) == 0.125
    assert V / (4 * 32) == 0.0625
    assert 64 / (2 * 32) == 1.0
    assert 128 / (4 * 32) == 1.0
    assert 256 / (2 * 32) == 4.0

    # 1-D fp32 vector tile.
    B, b, e = 256, 4, 4
    threads = B // e
    assert threads == 64 and threads == 2 * 32
    v = 128 // (8 * b)
    assert v == 4 and e == v
    assert e * b == 16
    assert 32 * e * b == 512
    assert cdiv(1024, B) == 4 and cdiv(1000, B) == 4

    # Double-buffer timeline.
    T = 4
    serial, pipelined = 2 * T, T + 1
    assert (serial, pipelined) == (8, 5)
    assert abs(serial / pipelined - 1.6) < 1e-12

    # Shared-memory panel pair, 32x32 fp16, two stages.
    assert 2 * (32 * 32 + 32 * 32) * 2 == 8192


test_worked_example()


## Part II — Method 01: scalar and tiled references

- **Connection.** Turns the Part I rules into a CPU oracle before any Triton launch.
- **Rule.** The scalar loop applies $y_i=\alpha x_i$ one lane at a time; the tiled loop walks the same tile map one Python slice per tile.
- **Where.** Correctness only: neither function is a performance baseline.
- **Triton bridge.** The tiled loop is the exact map that `scale1d`, `copy2d`, and `row_affine` reproduce on the GPU.


In [ ]:
def scalar_scale(x, alpha):
    """Method 01 — one lane at a time, exactly the definition."""
    flat_in = x.reshape(-1)
    flat_out = torch.empty_like(flat_in)
    for i in range(flat_in.numel()):
        flat_out[i] = flat_in[i] * alpha
    return flat_out.reshape(x.shape)


def tiled_scale_reference(x, BM, BN, alpha):
    """Method 01 — the same tile map the kernels use, expressed in Python slices."""
    M, N = x.shape
    out = torch.empty_like(x)
    live = []
    for pm in range(cdiv(M, BM)):
        for pn in range(cdiv(N, BN)):
            r0, c0 = pm * BM, pn * BN
            r1, c1 = min(r0 + BM, M), min(c0 + BN, N)
            out[r0:r1, c0:c1] = x[r0:r1, c0:c1] * alpha
            live.append((r1 - r0) * (c1 - c0))
    return out, live


def test_method01():
    """CPU — the scalar and tiled references agree, and the tile map is complete."""
    x = torch.arange(35, dtype=torch.float32).reshape(5, 7)

    out, live = tiled_scale_reference(x, 2, 4, 2.0)
    assert torch.equal(out, x * 2.0)
    assert live == [8, 6, 8, 6, 4, 3]
    assert sum(live) == 35

    assert torch.equal(scalar_scale(x, 2.0), x * 2.0)

    # A 1-D block of 4 over the flattened 35-element vector needs 9 programs.
    assert cdiv(x.numel(), 4) == 9
    out1, live1 = tiled_scale_reference(x, 1, 4, 0.5)
    assert torch.equal(out1, x * 0.5)
    assert sum(live1) == 35

    # Tail waste for the 2x4 tiling.
    assert abs((1 - 35 / 48) - 13 / 48) < 1e-12

    print("method 01: scalar and tiled references match; 35 cells covered once")


test_method01()


## Part III — Method 02: tiled Triton kernels

### 1-D tiled scale

- **Connection.** First Triton kernel; lifts the 19.1 mask and grid into `scale1d_kernel`.
- **Rule.** For $N=13$ and `BLOCK=4`, the grid is $P=4$; program 3 owns offsets $12,13,14,15$ and the mask keeps only $12$.
- **Example.** Tile map:

  | pid | base | offsets | mask | live |
  |---|---|---|---|---|
  | 0 | 0 | 0,1,2,3 | T T T T | 4 |
  | 1 | 4 | 4,5,6,7 | T T T T | 4 |
  | 2 | 8 | 8,9,10,11 | T T T T | 4 |
  | 3 | 12 | 12,13,14,15 | T F F F | 1 |

- **Where.** One load and one store; the template for every contiguous elementwise kernel.


In [ ]:
@triton.jit
def scale1d_kernel(x_ptr, out_ptr, n_elements, alpha, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x * alpha, mask=mask)


def scale1d(x, alpha, BLOCK=256, num_warps=2):
    x = x.contiguous()
    out = torch.empty_like(x)
    n = x.numel()
    assert n > 0
    grid = (triton.cdiv(n, BLOCK),)
    scale1d_kernel[grid](x, out, n, alpha, BLOCK=BLOCK, num_warps=num_warps)
    return out


def test_scale1d():
    require_gpu("1-D tiled scale")
    assert triton.cdiv(13, 4) == 4
    x = torch.randn(13, device=DEVICE)
    check("1-D scale N=13 BLOCK=4", scale1d(x, 2.0, BLOCK=4, num_warps=1), x * 2.0)

    assert triton.cdiv(10000, 256) == 40
    y = torch.randn(10000, device=DEVICE)
    check("1-D scale N=10000 BLOCK=256", scale1d(y, 1.5, BLOCK=256, num_warps=2), y * 1.5)


run_if_gpu(test_scale1d)


### 2-D tiled copy

- **Connection.** Adds the second axis from 19.2; the 1-D mask becomes an outer product.
- **Rule.** For the $5\times7$ matrix and a $2\times4$ tile, the grid is $(3,2)$.
- **Example.** Program $(1,0)$ owns rows $2,3$ and columns $0,1,2,3$; program $(2,1)$ owns row $4$ and columns $4,5,6$, with one masked row lane and one masked column lane.
- **Tile map (live cells).**

  | | $p_n=0$ | $p_n=1$ |
  |---|---|---|
  | $p_m=0$ | 8 | 6 |
  | $p_m=1$ | 8 | 6 |
  | $p_m=2$ | 4 | 3 |

- **Triton.** The mask is the outer product of the row mask and the column mask; both `tl.load` and `tl.store` carry it.


In [ ]:
@triton.jit
def copy2d_kernel(x_ptr, out_ptr, M, N, BM: tl.constexpr, BN: tl.constexpr):
    rows = tl.program_id(0) * BM + tl.arange(0, BM)
    cols = tl.program_id(1) * BN + tl.arange(0, BN)
    mask = (rows[:, None] < M) & (cols[None, :] < N)
    ptrs = rows[:, None] * N + cols[None, :]
    x = tl.load(x_ptr + ptrs, mask=mask, other=0.0)
    tl.store(out_ptr + ptrs, x * 2.0, mask=mask)


def copy2d(x, BM=8, BN=16, num_warps=4):
    M, N = x.shape
    out = torch.empty_like(x)
    grid = (triton.cdiv(M, BM), triton.cdiv(N, BN))
    copy2d_kernel[grid](x, out, M, N, BM=BM, BN=BN, num_warps=num_warps)
    return out


def test_copy2d():
    require_gpu("2-D tiled copy")
    x = torch.randn(5, 7, device=DEVICE)
    assert (triton.cdiv(5, 2), triton.cdiv(7, 4)) == (3, 2)
    check("2-D copy 5x7 tile 2x4", copy2d(x, 2, 4, num_warps=1), x * 2.0)

    y = torch.randn(37, 53, device=DEVICE)
    assert (triton.cdiv(37, 8), triton.cdiv(53, 16)) == (5, 4)
    check("2-D copy 37x53 tile 8x16", copy2d(y, 8, 16, num_warps=4), y * 2.0)


run_if_gpu(test_copy2d)


### Vectorized 4-elements-per-thread kernel

- **Connection.** Turns the 19.8 vector rule into a concrete launch.
- **Rule.** For `BLOCK=256`, `num_warps=2`, fp32: threads $=32\cdot2=64$, $e=256/64=4$, $v=16/4=4$.
- **Example.** Each lane owns four contiguous fp32 values, so the compiler can emit one 128-bit load and one 128-bit store per lane; the grid is $\lceil N/256\rceil$.

  | resource | value |
  |---|---:|
  | tile elements | 256 |
  | threads | 64 |
  | elements/thread | 4 |
  | bytes/thread | 16 |
  | bytes/warp | 512 |
  | grid for $N=1024$ | 4 |
  | grid for $N=1000$ | 4 |

- **Triton.** `tl.max_contiguous(tl.multiple_of(offs, 4), 4)` states the alignment and contiguity guarantee to the compiler.


In [ ]:
@triton.jit
def vec4_scale_kernel(x_ptr, out_ptr, n_elements, alpha, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    offs = tl.max_contiguous(tl.multiple_of(offs, 4), 4)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x * alpha, mask=mask)


def vec4_scale(x, alpha, BLOCK=256, num_warps=2):
    assert BLOCK % 4 == 0
    assert BLOCK == 32 * num_warps * 4, "this demo targets exactly 4 elements/thread"
    x = x.contiguous()
    out = torch.empty_like(x)
    n = x.numel()
    grid = (triton.cdiv(n, BLOCK),)
    vec4_scale_kernel[grid](x, out, n, alpha, BLOCK=BLOCK, num_warps=num_warps)
    return out


def test_vec4():
    require_gpu("vectorized 4-elements-per-thread kernel")
    assert 256 // (32 * 2) == 4
    assert 128 // (8 * 4) == 4
    assert 4 * 4 == 16
    assert 32 * 16 == 512

    x = torch.randn(1024, device=DEVICE)
    assert triton.cdiv(1024, 256) == 4
    check("vec4 scale N=1024", vec4_scale(x, 3.0), x * 3.0)

    y = torch.randn(1000, device=DEVICE)
    assert triton.cdiv(1000, 256) == 4
    check("vec4 scale N=1000 tail", vec4_scale(y, 0.5), y * 0.5)


run_if_gpu(test_vec4)


## Part IV — Method 03: register blocking, reuse, and pipelining

### Register-blocked row kernel

- **Connection.** Combines the 19.9 register block with the row structure from 19.2.
- **Rule.** The row-wise affine map is $y_{m,n}=s_mx_{m,n}+b_m$; each program owns `ROWS` rows and `BLOCK_N` columns.
- **Example.** With `ROWS=4`, `BLOCK_N=64`, and `num_warps=2`: program tile $4\cdot64=256$ elements, $256/64=4$ elements per thread — a $4\times1$ register block per thread in the column direction, plus the row loop.
- **Reuse.** The scale and bias are loaded once per row and broadcast across the column tile: scale reads $=M$, bias reads $=M$, saved reads $=2M(N-1)$.
- **Numbers.** For $M=37$, $N=53$, fp32, naive scale/bias traffic would be $2\cdot37\cdot53\cdot4=15{,}688$ bytes; the tiled version reads $2\cdot37\cdot4=296$ bytes.
- **Where.** Any per-row normalization, scale, bias, or activation that is constant across columns.


In [ ]:
@triton.jit
def row_affine_kernel(x_ptr, scale_ptr, bias_ptr, out_ptr, M, N,
                      ROWS: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    rows = pid_m * ROWS + tl.arange(0, ROWS)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    row_mask = rows < M
    col_mask = cols < N
    mask = row_mask[:, None] & col_mask[None, :]

    ptrs = rows[:, None] * N + cols[None, :]
    x = tl.load(x_ptr + ptrs, mask=mask, other=0.0)
    s = tl.load(scale_ptr + rows, mask=row_mask, other=1.0)
    b = tl.load(bias_ptr + rows, mask=row_mask, other=0.0)

    tl.store(out_ptr + ptrs, x * s[:, None] + b[:, None], mask=mask)


def row_affine(x, scale, bias, ROWS=4, BLOCK_N=64, num_warps=2):
    M, N = x.shape
    out = torch.empty_like(x)
    grid = (triton.cdiv(M, ROWS), triton.cdiv(N, BLOCK_N))
    row_affine_kernel[grid](x, scale, bias, out, M, N,
                            ROWS=ROWS, BLOCK_N=BLOCK_N, num_warps=num_warps)
    return out


def test_row_affine():
    require_gpu("register-blocked row kernel")
    torch.manual_seed(0)
    x = torch.randn(37, 53, device=DEVICE)
    scale = torch.randn(37, device=DEVICE)
    bias = torch.randn(37, device=DEVICE)
    ref = x * scale[:, None] + bias[:, None]

    assert (triton.cdiv(37, 4), triton.cdiv(53, 64)) == (10, 1)
    assert (4 * 64) / (32 * 2) == 4.0
    check("row affine 4x64 tile, 2 warps", row_affine(x, scale, bias, 4, 64, 2), ref)

    assert (4 * 32) / (32 * 2) == 2.0
    check("row affine 4x32 tile, 2 warps", row_affine(x, scale, bias, 4, 32, 2), ref)


run_if_gpu(test_row_affine)


### Shared-memory-style reuse

- **Connection.** Lifts the 19.9 per-thread reuse to the block scope introduced in 19.10.
- **Hardware.** Shared memory is a per-block scratchpad with the lifetime of one program; a tile loaded into it can be read by every thread in the block.
- **Registers.** Registers are per-thread; a value loaded into a register can be reused by every output that thread owns.
- **Triton.** `tl.dot` stages panels through shared memory, and the pipeline stages multiply the panel buffers; Triton does not expose a user-managed shared array.
- **Broadcast tiles.** The elementwise version of the same idea: load a row vector of length $B_M$ once and reuse it across $B_N$ columns.
- **Example.** $M=37$, $N=53$, $B_N=64$, fp32: naive row reads $=37\cdot53\cdot4=7844$ B; reused row reads $=37\cdot4=148$ B.
- **Numbers.** The ratio is $53\times$; the useful arithmetic is unchanged.
- **Bridge.** A staging buffer is only useful when the next load overlaps the current compute.


In [ ]:
@triton.jit
def row_broadcast_kernel(x_ptr, row_ptr, out_ptr, M, N,
                         BM: tl.constexpr, BN: tl.constexpr):
    rows = tl.program_id(0) * BM + tl.arange(0, BM)
    cols = tl.program_id(1) * BN + tl.arange(0, BN)
    row_mask = rows < M
    col_mask = cols < N
    mask = row_mask[:, None] & col_mask[None, :]

    ptrs = rows[:, None] * N + cols[None, :]
    x = tl.load(x_ptr + ptrs, mask=mask, other=0.0)
    r = tl.load(row_ptr + rows, mask=row_mask, other=0.0)
    tl.store(out_ptr + ptrs, x + r[:, None], mask=mask)


def row_broadcast(x, row, BM=4, BN=64, num_warps=2):
    M, N = x.shape
    out = torch.empty_like(x)
    grid = (triton.cdiv(M, BM), triton.cdiv(N, BN))
    row_broadcast_kernel[grid](x, row, out, M, N, BM=BM, BN=BN, num_warps=num_warps)
    return out


def test_reuse():
    require_gpu("shared-memory-style reuse")
    x = torch.randn(37, 53, device=DEVICE)
    r = torch.randn(37, device=DEVICE)
    check("row broadcast reuse", row_broadcast(x, r, 4, 64, 2), x + r[:, None])

    M, N, b = 37, 53, 4
    naive = M * N * b
    reused = M * b * triton.cdiv(N, 64)
    assert naive == 7844
    assert reused == 148
    assert naive / reused == 53.0
    print("reuse: row traffic 148 B instead of 7844 B (53x)")


run_if_gpu(test_reuse)


### Double buffering and a two-stage pipeline

- **Connection.** Implements the 19.11/19.12 schedule as a manual look-ahead loop.
- **Rule.** Load panel $k+1$ before storing panel $k$; the last panel is stored after the loop.
- **Schedule.**

  ```text
  prologue:    cur = load(panel 0)
  iteration k: nxt = load(panel k+1)
               store(panel k)
               cur = nxt
  epilogue:    store(last panel)
  ```

- **Triton.** This is what `num_stages=2` asks the compiler to build automatically; the result must be identical for `num_stages=1,2,3`.
- **Example.** With `BLOCK=64`, `NSTEPS=4`, and $N=1000$, each program owns $4\cdot64=256$ elements, so the grid is $\lceil1000/256\rceil=4$.
- **Model.** Serial $2T=8$ units; two-stage $T+1=5$; a $1.6\times$ schedule improvement.


In [ ]:
@triton.jit
def pipeline_kernel(x_ptr, out_ptr, n_elements, alpha,
                    BLOCK: tl.constexpr, NSTEPS: tl.constexpr):
    pid = tl.program_id(0)
    base = pid * NSTEPS * BLOCK

    offs = base + tl.arange(0, BLOCK)
    cur = tl.load(x_ptr + offs, mask=offs < n_elements, other=0.0)

    for k in range(1, NSTEPS):
        nxt_offs = base + k * BLOCK + tl.arange(0, BLOCK)
        nxt = tl.load(x_ptr + nxt_offs, mask=nxt_offs < n_elements, other=0.0)
        tl.store(out_ptr + offs, cur * alpha, mask=offs < n_elements)
        offs = nxt_offs
        cur = nxt

    tl.store(out_ptr + offs, cur * alpha, mask=offs < n_elements)


def pipeline_scale(x, alpha, BLOCK=64, NSTEPS=4, num_stages=2):
    x = x.contiguous()
    out = torch.empty_like(x)
    n = x.numel()
    grid = (triton.cdiv(n, BLOCK * NSTEPS),)
    pipeline_kernel[grid](x, out, n, alpha, BLOCK=BLOCK, NSTEPS=NSTEPS,
                          num_stages=num_stages)
    return out


def test_pipeline():
    require_gpu("two-stage software pipeline")
    torch.manual_seed(0)
    x = torch.randn(1000, device=DEVICE)
    ref = x * 1.25
    assert triton.cdiv(1000, 64 * 4) == 4

    for stages in (1, 2, 3):
        torch.testing.assert_close(
            pipeline_scale(x, 1.25, BLOCK=64, NSTEPS=4, num_stages=stages), ref
        )

    T = 4
    assert (2 * T, T + 1) == (8, 5)
    assert abs((2 * T) / (T + 1) - 1.6) < 1e-12
    print("pipeline: stages 1,2,3 agree; schedule model 8 -> 5 steps")


run_if_gpu(test_pipeline)


## Choosing tile, block, and warps

- **Connection.** Closes the Part I ladder by turning the five quantities into a selection procedure.
- **Tail waste.** $w=1-\frac{MN}{PV}$; a $5\times7$ matrix with $2\times4$ tiles has $w=1-35/48=27.1\%$, while $64\times64$ with $32\times32$ has $w=0$.
- **Elements per thread.** $e=V/(32W)\ge1$ and $e\ge v=16/b$; fractional $e$ means idle lanes, very large $e$ means register pressure.
- **Register pressure.** A rough count is $R\approx e+r_mr_n$; if $R$ exceeds the per-thread budget, the compiler spills to local memory.
- **Occupancy.** A rough bound is $O\approx\min(1,R_{\max}/R,Q_{\text{smem}}/(sQ_{\text{stage}}),W_{\max}/W)$; it names the likely limiter, not a measurement.
- **Trade-off table.**

  | regime | tile $V$ | warps $W$ | $e$ | reuse | occupancy | tail waste | verdict |
  |---|---:|---:|---:|---|---|---|---|
  | too small | 8 | 2 | 0.125 | none | high | 27% (5×7) | lanes idle |
  | vectorized | 256 | 2 | 4 | low | high | shape-dependent | one 16-B vector/lane |
  | balanced | 1024 | 4 | 8 | medium | medium | low | good default for 2-D |
  | too large | 4096 | 8 | 16 | high | low | low | spill risk |

- **Rule of thumb.** Choose the largest power-of-two tile that keeps $e$ in $[1,8]$, keeps the tile bytes inside the register/shared-memory budget, and keeps tail waste acceptable; then measure.


In [ ]:
def tail_waste(M, N, BM, BN):
    P = cdiv(M, BM) * cdiv(N, BN)
    return 1.0 - (M * N) / (P * BM * BN)


def elements_per_thread(V, W):
    return V / (32 * W)


def test_choice_table():
    assert abs(tail_waste(5, 7, 2, 4) - 13 / 48) < 1e-12
    assert tail_waste(64, 64, 32, 32) == 0.0
    assert abs(tail_waste(65, 65, 32, 32) - (1 - 65 * 65 / (3 * 3 * 1024))) < 1e-12

    assert elements_per_thread(8, 2) == 0.125
    assert elements_per_thread(256, 2) == 4.0
    assert elements_per_thread(1024, 4) == 8.0
    assert elements_per_thread(4096, 8) == 16.0

    widths = {1: 16, 2: 8, 4: 4, 8: 2}
    for b, v in widths.items():
        assert 128 // (8 * b) == v
        assert v * b == 16

    regs_max, regs_used = 255, 64
    smem_max, smem_used = 49152, 8192
    warps_max, warps_used = 64, 8
    occupancy = min(1.0, regs_max / regs_used, smem_max / smem_used,
                    warps_max / warps_used)
    assert occupancy == 1.0
    assert min(1.0, regs_max / 300) < 1.0

    print("choice table: tail waste, e/thread, vector width, occupancy bound verified")


test_choice_table()


## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| Tile too small | $e\lt 1$; most lanes idle | make $V\ge32W$; enlarge the tile or reduce `num_warps` |
| Tile not a power of two | `tl.arange` rejects the extent | use $1,2,4,\dots$ for every block size |
| Tail waste ignored | up to almost half the lanes masked on small shapes | choose tiles that divide the shape, or accept and measure $w$ |
| Too many warps | $e=V/(32W)$ drops below 1 | lower `num_warps` or enlarge the tile |
| Register spill | local-memory traffic replaces reuse | lower $e$, $r_mr_n$, or the tile volume |
| Unaligned vector load | one 16-B request splits into several transactions | align the base and use `tl.multiple_of` / `tl.max_contiguous` |
| Pipeline stage mismatch | buffers exceed the loop trip count or shared memory | use `num_stages <= number of panels`; check the shared-memory budget |
| Masking one axis only | out-of-range reads on the other axis | combine `rows[:, None] < M` with `cols[None, :] < N` |
| Integer offset overflow | wrong addresses for large batches | promote to `int64` before `b*M*N` |
| Measuring on the wrong device | a schedule that helps one GPU hurts another | benchmark on the target; report hardware with any speedup |


## Self-check

1. For the $5\times7$ matrix and $2\times4$ tiles, write the six tile coordinates and the live-cell count of each. Which tile has the most masked lanes?
2. Compute $P$, $V$, and $e$ for a $129\times257$ matrix with $16\times32$ tiles and 4 warps.
3. Why is $e=0.125$ a problem even though the kernel still produces the right answer?
4. For fp16 and fp32, what are the vector widths and the bytes per lane? Why are they both 16 bytes?
5. A thread owns a $4\times4$ register block. How many accumulators does it hold, and how many outputs does one loaded $A$ value feed?
6. With $T=8$ panels and two stages, what are the serial and pipelined step counts? What is the model speedup?
7. Why does `num_stages=3` not change the numerical result of the pipeline kernel?
8. Triton does not expose shared memory directly. What does `tl.dot` use it for, and how does that show up in the `num_stages` cost?


## Summary

- **Tiling** chooses the unit of work; **vectorization** makes that unit's traffic wide and aligned.
- **The ladder:** 1-D/2-D/3-D tiles → block size → tile size → warps → elements/thread → vector width → register blocking → shared-memory reuse → double buffering → software pipelining.
- **Correctness** is the mask; **quality** is $e$ in $[1,8]$, $e\ge v$, and a tile that fits the register/shared-memory budget.
- **The numbers** in this chapter are counts and schedules, not measurements; any speedup claim needs the target hardware.

## Key takeaways

- Grid, volume, and elements/thread are three one-line formulas; compute them before writing a kernel.
- A tile smaller than its warp complement wastes lanes even when it is correct.
- A tile larger than the register budget spills even when it is correct.
- Contiguous, aligned offsets are what turn a load into a vector load.
- Reuse is the point of a larger tile; measure whether the reuse pays for the occupancy it costs.
- Pipeline depth changes the schedule, not the answer.

## Checkpoints

- [ ] Can you compute the grid and the tile map for an arbitrary shape and tile?
- [ ] Can you derive $e=V/(32W)$ and say whether a tile fills its warps?
- [ ] Can you pick a vector width from the dtype and check the 16-byte limit?
- [ ] Can you explain the difference between a block size and a tile size?
- [ ] Can you explain why a larger tile can be slower despite doing more reuse?
- [ ] Can you write a two-stage look-ahead loop and verify it against a serial one?

**Where this leads.** Chapter 20 fuses these tiles into single kernels; the same grid, mask, and vector rules decide whether fusion saves traffic or just adds register pressure.


## Exercises

1. **Tile map by hand.** For $M=7$, $N=11$, $B_M=3$, $B_N=4$, list every tile, its live cells, and the tail waste. Check your table against `tile_map` and `tail_waste`.
2. **Warp arithmetic.** For $V\in\{64,128,256,512\}$ and $W\in\{1,2,4,8\}$, tabulate $e$. Mark the entries below 1 and above 16.
3. **1-D sweep.** Run `scale1d` for `BLOCK` in $\{64,128,256,512\}$ and `num_warps` in $\{1,2,4\}$. Keep only the combinations with $e\ge1$; explain the pattern.
4. **2-D tail.** Change the $37\times53$ copy test to a $37\times49$ shape with `BLOCK_N=16`. Which column block has the most masked lanes?
5. **Vector width.** Rewrite `vec4_scale_kernel` for fp16 so that each lane still moves 16 bytes. What is the new elements/thread and the new `BLOCK` for 2 warps?
6. **Register block.** Extend `row_affine_kernel` so each program owns 8 rows instead of 4. Compute the new $e$ for `BLOCK_N=64` and 2 warps, then check the result.
7. **Shared-memory budget.** For $B_M=B_N=B_K=64$, fp16, how many bytes per stage? How many stages fit in 48 KiB?
8. **Pipeline depth.** Run the pipeline kernel with `NSTEPS=1,2,4,8`. Which values are degenerate, and why?
9. **Alignment.** Offset a 1-D tensor by one element so the base is no longer 16-byte aligned. Does the result stay correct? What did you lose?
10. **Roofline.** For a $4096\times4096$ fp32 copy, compute the bytes moved and the minimum time at 1 TB/s. Compare with a tiled version that reads and writes the same bytes; explain why tiling alone cannot beat that bound.


In [ ]:
CPU_TESTS = [
    test_worked_example,
    test_3d_tile_map,
    test_method01,
    test_choice_table,
]

GPU_TESTS = [
    test_scale1d,
    test_copy2d,
    test_vec4,
    test_row_affine,
    test_reuse,
    test_pipeline,
]

print("CPU tests")
for fn in CPU_TESTS:
    fn()

print("\nGPU tests")
for fn in GPU_TESTS:
    run_if_gpu(fn)

print("\ndone: CPU checks passed; GPU checks ran or printed SKIP")
